<div style="background-color: #f8fafc; border-left: 6px solid #0d9488; padding: 20px 24px; border-radius: 6px; box-shadow: 0 1px 3px rgba(0,0,0,0.06); font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, Arial, sans-serif;">
  <span style="font-size: 11px; font-weight: 700; text-transform: uppercase; letter-spacing: 1.5px; color: #0d9488;">AIML Honors &bull; Lab Assignment 04</span>
  <h1 style="margin: 8px 0 12px 0; color: #0f172a; font-size: 24px; font-weight: 700;">Employee Data Management & Analytics with Pandas</h1>
  <div style="font-size: 14px; color: #475569; line-height: 1.6;">
    <strong>Student Name:</strong> Pranjal Khadke &nbsp;|&nbsp; 
    <strong>Email:</strong> <a href="mailto:Pranjalckhadke@gmail.com" style="color: #0d9488; text-decoration: none;">Pranjalckhadke@gmail.com</a> &nbsp;|&nbsp; 
    <strong>GitHub:</strong> <a href="https://github.com/PranjalKhadke" style="color: #0d9488; text-decoration: none;">github.com/PranjalKhadke</a><br>
    <strong>Course:</strong> AIML Honors &bull; Sem III &nbsp;|&nbsp; 
    <strong>Topic:</strong> Exploratory Data Analysis, Vectorized Masking & GroupBy Aggregations
  </div>
</div>

### 1. Aim & Theoretical Foundations

**Aim:** To harness the `pandas` analytical framework for enterprise workforce modeling, demonstrating in-memory two-dimensional data structures (`DataFrame`), vectorized boolean indexing, label-based cell manipulation (`.loc`), multi-column `groupby()` aggregations, and disk serialization.

#### Computational Concepts Explored:
1. **Contiguous Memory Storage & Vectorization:** Pandas leverages C-aligned NumPy ndarrays under the hood, enabling SIMD (Single Instruction Multiple Data) filtering and aggregations that run orders of magnitude faster than naive Python `for` loops.
2. **Boolean Masking:** Evaluating logical expressions like `(df['salary'] >= 85000) & (df['experience_yrs'] >= 3.0)` yields a boolean Series that indexes records in $O(N)$ vectorized time.
3. **Split-Apply-Combine Paradigm:** `df.groupby('department').agg(...)` splits the workforce by operational units, computes aggregations independently, and combines the summaries into a clean reporting table.

#### Attribute Schema:
| Column Name | Type | Analytical Role |
| :--- | :--- | :--- |
| `emp_id` | `int64` | Primary key identifier |
| `name` | `object` (str) | Employee identity |
| `department` | `object` (str) | Categorical grouping variable |
| `experience_yrs` | `float64` | Quantitative predictor (Years of Experience) |
| `salary` | `float64` | Target quantitative metric (Monthly Compensation in INR) |


In [1]:
import pandas as pd
import os

CSV_PATH = 'employee.csv'
COLUMNS = ['emp_id', 'name', 'department', 'experience_yrs', 'salary']
print('[PANDAS ENGINE] Library loaded successfully. Ready for workforce analytics.')

[PANDAS ENGINE] Library loaded successfully. Ready for workforce analytics.


In [2]:
# Initializing workforce cohort
cohort_data = [
    {'emp_id': 301, 'name': 'Pranjal Khadke', 'department': 'AI Research', 'experience_yrs': 3.5, 'salary': 92000.0},
    {'emp_id': 302, 'name': 'Kavita Deshmukh', 'department': 'Data Science', 'experience_yrs': 4.0, 'salary': 85000.0},
    {'emp_id': 303, 'name': 'Arjun Nair', 'department': 'Engineering', 'experience_yrs': 2.0, 'salary': 68000.0},
    {'emp_id': 304, 'name': 'Pooja Iyer', 'department': 'Product', 'experience_yrs': 5.0, 'salary': 96000.0},
    {'emp_id': 305, 'name': 'Sameer Kulkarni', 'department': 'Engineering', 'experience_yrs': 1.5, 'salary': 62000.0},
    {'emp_id': 306, 'name': 'Meera Joshi', 'department': 'AI Research', 'experience_yrs': 3.0, 'salary': 89000.0}
]

df = pd.DataFrame(cohort_data, columns=COLUMNS)
print(f'[DATA LOAD] Initialized DataFrame with {len(df)} employee observations across {df["department"].nunique()} business departments.')

[DATA LOAD] Initialized DataFrame with 6 employee observations across 4 business departments.


In [3]:
# Tabular view of complete DataFrame
df

,emp_id,name,department,experience_yrs,salary
0,301,Pranjal Khadke,AI Research,3.5,92000.0
1,302,Kavita Deshmukh,Data Science,4.0,85000.0
2,303,Arjun Nair,Engineering,2.0,68000.0
3,304,Pooja Iyer,Product,5.0,96000.0
4,305,Sameer Kulkarni,Engineering,1.5,62000.0
5,306,Meera Joshi,AI Research,3.0,89000.0


In [4]:
# Parametric statistical profiling of numeric features
df[['experience_yrs', 'salary']].describe().T[['count', 'mean', 'std', 'min', 'max']]

                count          mean           std      min      max
experience_yrs    6.0      3.166667      1.329160      1.5      5.0
salary            6.0  82000.000000  14156.270695  62000.0  96000.0

In [5]:
# Split-Apply-Combine: Multi-column GroupBy summary
dept_analytics = df.groupby('department').agg(
    headcount=('emp_id', 'count'),
    avg_experience=('experience_yrs', 'mean'),
    avg_salary=('salary', 'mean'),
    total_payroll=('salary', 'sum')
)
dept_analytics

              headcount  avg_experience  avg_salary  total_payroll
department                                                        
AI Research           2            3.25     90500.0       181000.0
Data Science          1            4.00     85000.0        85000.0
Engineering           2            1.75     65000.0       130000.0
Product               1            5.00     96000.0        96000.0

In [6]:
# Vectorized boolean indexing: Senior personnel (salary >= 85000 AND exp >= 3.0 yrs)
senior_filter = (df['salary'] >= 85000) & (df['experience_yrs'] >= 3.0)
senior_cohort = df[senior_filter]
senior_cohort

   emp_id            name   department  experience_yrs   salary
0     301  Pranjal Khadke  AI Research             3.5  92000.0
1     302 Kavita Deshmukh Data Science             4.0  85000.0
3     304      Pooja Iyer      Product             5.0  96000.0

In [7]:
# In-place cell modification utilizing label-based .loc indexing
df.loc[df['emp_id'] == 305, 'salary'] = 70000.0
df.loc[df['emp_id'] == 305, 'experience_yrs'] = 2.0
print('[UPDATE] Sameer Kulkarni (ID 305): Promoted with compensation updated to Rs. 70,000.00.')

[UPDATE] Sameer Kulkarni (ID 305): Promoted with compensation updated to Rs. 70,000.00.


In [8]:
# Record expunction utilizing boolean inversion
df = df[df['emp_id'] != 303].reset_index(drop=True)
print(f'[DELETE] Discharged Employee ID 303. Active staff count: {len(df)}')

[DELETE] Discharged Employee ID 303. Active staff count: 5


In [9]:
# Disk persistence and roundtrip verification
df.to_csv(CSV_PATH, index=False)
print(f"[SERIALIZATION] Exported {len(df)} records to '{CSV_PATH}'.")
reloaded_df = pd.read_csv(CSV_PATH)
print(f'[INTEGRITY CHECK] Verified reload from disk: {len(reloaded_df)} rows, {len(reloaded_df.columns)} features match.')

[SERIALIZATION] Exported 5 records to 'employee.csv'.
[INTEGRITY CHECK] Verified reload from disk: 5 rows, 5 features match.


<div style="background-color: #f0fdf4; border: 1px solid #bbf7d0; border-left: 5px solid #16a34a; padding: 16px 20px; border-radius: 6px; font-family: sans-serif;">
  <h4 style="margin: 0 0 6px 0; color: #15803d; font-size: 15px; font-weight: 700;">Conclusion & Key Analytical Findings</h4>
  <p style="margin: 0; color: #166534; font-size: 13.5px; line-height: 1.6;">
    The Pandas framework demonstrated superior analytical expressiveness compared to procedural CSV and basic SQL scripts. Multi-metric aggregations (mean, count, sum) were computed in single concise expressions. The <code>.loc</code> indexer guaranteed safe, in-place cell updates without risking SettingWithCopy warnings, and disk serialization to CSV ensured seamless interoperability.
  </p>
</div>